# 3D数据精简工具

**目的**: 从完整的3D数据中提取训练所需的3个关键key，减少文件大小和IO负担

**提取的key**:
- `data`: (384, 336, 256, 351) - 多模态特征
- `region_mask`: (384, 336, 256) - ROI掩膜
- `region_labels`: (384, 336, 256) - 102类脑区域标签

**优化**:
- 使用低压缩级别（compression_opts=1）加快机械硬盘IO
- 顺序处理避免随机访问
- 断点续传支持
- 只读原始文件，安全可靠

## 1. 导入库和配置

In [ ]:
import h5py
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
import json
from datetime import datetime
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("✅ 库导入成功")

In [ ]:
# ==================== 配置区域 ====================
# 根据运行环境修改这里的路径

# 服务器路径（默认）
BASE_DIR = "/home/jovyan/gpu_space/workspace_jiayi/alex_datasets"

# 本地测试路径（取消注释以使用）
# BASE_DIR = "/Users/jannik/path/to/local/data"

# 输入输出目录
INPUT_3D_DIR = Path(BASE_DIR) / "3D_validated"
OUTPUT_3D_MINIMAL_DIR = Path(BASE_DIR) / "3D_minimal"

# 必需的key列表
REQUIRED_KEYS = ['data', 'region_mask', 'region_labels']

# 预期的数据维度
EXPECTED_SHAPES = {
    'data': (351, 384, 336, 256),  # HDF5格式，特征维在前
    'region_mask': (384, 336, 256),
    'region_labels': (384, 336, 256)
}

print(f"📂 输入目录: {INPUT_3D_DIR}")
print(f"📂 输出目录: {OUTPUT_3D_MINIMAL_DIR}")
print(f"🔑 提取的key: {REQUIRED_KEYS}")

## 2. 扫描和验证文件

In [ ]:
# 检查输入目录是否存在
if not INPUT_3D_DIR.exists():
    raise FileNotFoundError(f"❌ 输入目录不存在: {INPUT_3D_DIR}")

# 创建输出目录
OUTPUT_3D_MINIMAL_DIR.mkdir(parents=True, exist_ok=True)
print(f"✅ 输出目录已创建: {OUTPUT_3D_MINIMAL_DIR}")

# 扫描所有.mat文件
mat_files = sorted(list(INPUT_3D_DIR.glob("*.mat")))

if len(mat_files) == 0:
    raise ValueError(f"❌ 未找到任何.mat文件在: {INPUT_3D_DIR}")

print(f"\n📊 找到 {len(mat_files)} 个.mat文件")
print("\n前5个文件示例:")
for i, f in enumerate(mat_files[:5], 1):
    file_size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {i}. {f.name} ({file_size_mb:.1f} MB)")

## 3. 核心处理函数

In [ ]:
def generate_output_filename(input_filename: str) -> str:
    """
    生成输出文件名
    例如: PDP_02_xxx_3d_validated.mat -> PDP_02_xxx_3d_validated_minimal.mat
    """
    if input_filename.endswith('.mat'):
        return input_filename[:-4] + '_minimal.mat'
    else:
        return input_filename + '_minimal'


def validate_mat_keys(h5_file) -> tuple:
    """
    验证MAT文件是否包含必需的key
    返回: (is_valid, missing_keys, existing_keys)
    """
    existing_keys = [k for k in h5_file.keys() if not k.startswith('#')]
    missing_keys = [k for k in REQUIRED_KEYS if k not in existing_keys]
    is_valid = len(missing_keys) == 0
    return is_valid, missing_keys, existing_keys


def validate_data_shapes(h5_file) -> tuple:
    """
    验证数据维度是否正确
    返回: (is_valid, shape_info)
    """
    shape_info = {}
    is_valid = True
    
    for key in REQUIRED_KEYS:
        if key in h5_file:
            actual_shape = h5_file[key].shape
            expected_shape = EXPECTED_SHAPES.get(key)
            shape_info[key] = {
                'actual': actual_shape,
                'expected': expected_shape,
                'match': actual_shape == expected_shape
            }
            if not shape_info[key]['match']:
                is_valid = False
    
    return is_valid, shape_info


def extract_minimal_data(input_path: Path, output_path: Path, verify_shapes: bool = True) -> dict:
    """
    从完整3D数据中提取精简版本
    
    Args:
        input_path: 输入MAT文件路径
        output_path: 输出MAT文件路径
        verify_shapes: 是否验证数据维度
    
    Returns:
        result: 处理结果字典
    """
    result = {
        'input_file': input_path.name,
        'output_file': output_path.name,
        'status': 'unknown',
        'error': None
    }
    
    try:
        # 打开输入文件
        with h5py.File(input_path, 'r') as f_in:
            # 验证key
            is_valid, missing_keys, existing_keys = validate_mat_keys(f_in)
            result['existing_keys'] = existing_keys
            
            if not is_valid:
                result['status'] = 'failed'
                result['error'] = f"缺少必需的key: {missing_keys}"
                return result
            
            # 验证维度（可选）
            if verify_shapes:
                shapes_valid, shape_info = validate_data_shapes(f_in)
                result['shape_info'] = shape_info
                if not shapes_valid:
                    result['status'] = 'warning'
                    result['error'] = "数据维度与预期不符，但仍继续处理"
            
            # 创建输出文件并复制数据
            with h5py.File(output_path, 'w') as f_out:
                for key in REQUIRED_KEYS:
                    # 读取数据
                    data = f_in[key][:]
                    
                    # 保存数据（使用低压缩级别，适合机械硬盘）
                    f_out.create_dataset(
                        key,
                        data=data,
                        compression='gzip',
                        compression_opts=1  # 低压缩，快速IO
                    )
        
        # 记录文件大小
        result['input_size_mb'] = input_path.stat().st_size / (1024 * 1024)
        result['output_size_mb'] = output_path.stat().st_size / (1024 * 1024)
        result['saved_mb'] = result['input_size_mb'] - result['output_size_mb']
        result['compression_ratio'] = result['output_size_mb'] / result['input_size_mb']
        
        if result['status'] == 'unknown':
            result['status'] = 'success'
        
    except Exception as e:
        result['status'] = 'failed'
        result['error'] = str(e)
    
    return result


print("✅ 处理函数定义完成")

## 4. 批量处理主程序

In [ ]:
# 初始化结果列表
results = []
success_count = 0
failed_count = 0
skipped_count = 0

# 处理每个文件
print(f"\n🚀 开始批量处理 {len(mat_files)} 个文件...\n")

for input_path in tqdm(mat_files, desc="处理进度"):
    # 生成输出文件名
    output_filename = generate_output_filename(input_path.name)
    output_path = OUTPUT_3D_MINIMAL_DIR / output_filename
    
    # 检查是否已存在（断点续传）
    if output_path.exists():
        tqdm.write(f"⏭️  跳过已存在: {input_path.name}")
        skipped_count += 1
        continue
    
    # 处理文件
    result = extract_minimal_data(input_path, output_path, verify_shapes=True)
    results.append(result)
    
    # 统计
    if result['status'] == 'success':
        success_count += 1
        tqdm.write(f"✅ {input_path.name}: {result['input_size_mb']:.1f}MB → {result['output_size_mb']:.1f}MB (节省 {result['saved_mb']:.1f}MB)")
    elif result['status'] == 'warning':
        success_count += 1
        tqdm.write(f"⚠️  {input_path.name}: {result['error']}")
    else:
        failed_count += 1
        tqdm.write(f"❌ {input_path.name}: {result['error']}")

print(f"\n{'='*60}")
print(f"📊 处理完成统计")
print(f"{'='*60}")
print(f"总文件数: {len(mat_files)}")
print(f"成功处理: {success_count}")
print(f"处理失败: {failed_count}")
print(f"跳过已存在: {skipped_count}")
print(f"{'='*60}")

## 5. 生成详细报告

In [ ]:
if results:
    # 创建DataFrame
    df = pd.DataFrame(results)
    
    # 显示统计摘要
    print("\n📈 文件大小统计")
    print("="*60)
    
    successful_results = df[df['status'].isin(['success', 'warning'])]
    
    if len(successful_results) > 0:
        total_input_size = successful_results['input_size_mb'].sum()
        total_output_size = successful_results['output_size_mb'].sum()
        total_saved = successful_results['saved_mb'].sum()
        avg_compression = successful_results['compression_ratio'].mean()
        
        print(f"原始总大小: {total_input_size:.1f} MB")
        print(f"精简后总大小: {total_output_size:.1f} MB")
        print(f"节省空间: {total_saved:.1f} MB ({(1-avg_compression)*100:.1f}%)")
        print(f"平均压缩比: {avg_compression:.2%}")
        print("="*60)
        
        # 保存CSV报告
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        csv_path = OUTPUT_3D_MINIMAL_DIR / f"extraction_report_{timestamp}.csv"
        df.to_csv(csv_path, index=False)
        print(f"\n📄 详细报告已保存: {csv_path}")
        
        # 保存JSON报告
        json_report = {
            'timestamp': timestamp,
            'summary': {
                'total_files': len(mat_files),
                'success': success_count,
                'failed': failed_count,
                'skipped': skipped_count,
                'total_input_mb': float(total_input_size),
                'total_output_mb': float(total_output_size),
                'total_saved_mb': float(total_saved),
                'avg_compression_ratio': float(avg_compression)
            },
            'details': results
        }
        
        json_path = OUTPUT_3D_MINIMAL_DIR / f"extraction_report_{timestamp}.json"
        with open(json_path, 'w') as f:
            json.dump(json_report, f, indent=2, default=str)
        print(f"📄 JSON报告已保存: {json_path}")
        
        # 显示处理结果表格
        print("\n📋 处理详情（前10条）:")
        display_df = df[['input_file', 'status', 'input_size_mb', 'output_size_mb', 'saved_mb']].head(10)
        display_df.columns = ['文件名', '状态', '原始大小(MB)', '精简后(MB)', '节省(MB)']
        print(display_df.to_string(index=False))
    else:
        print("⚠️ 没有成功处理的文件")
else:
    print("⚠️ 没有处理任何新文件（可能都已存在）")

## 6. 验证精简后数据的完整性（可选）

In [ ]:
def verify_minimal_file(original_path: Path, minimal_path: Path) -> dict:
    """
    验证精简文件与原始文件的数据一致性
    """
    verification_result = {
        'file': minimal_path.name,
        'all_match': True,
        'details': {}
    }
    
    try:
        with h5py.File(original_path, 'r') as f_orig, h5py.File(minimal_path, 'r') as f_min:
            for key in REQUIRED_KEYS:
                orig_data = f_orig[key][:]
                min_data = f_min[key][:]
                
                # 检查形状
                shape_match = orig_data.shape == min_data.shape
                
                # 检查数值
                data_match = np.array_equal(orig_data, min_data)
                
                verification_result['details'][key] = {
                    'shape_match': shape_match,
                    'data_match': data_match,
                    'original_shape': orig_data.shape,
                    'minimal_shape': min_data.shape
                }
                
                if not (shape_match and data_match):
                    verification_result['all_match'] = False
    
    except Exception as e:
        verification_result['all_match'] = False
        verification_result['error'] = str(e)
    
    return verification_result


# 验证前3个文件作为抽样检查
print("\n🔍 抽样验证数据一致性（前3个文件）...\n")

verification_results = []
for i, input_path in enumerate(mat_files[:3]):
    output_filename = generate_output_filename(input_path.name)
    output_path = OUTPUT_3D_MINIMAL_DIR / output_filename
    
    if output_path.exists():
        result = verify_minimal_file(input_path, output_path)
        verification_results.append(result)
        
        if result['all_match']:
            print(f"✅ {result['file']}: 数据完全一致")
        else:
            print(f"❌ {result['file']}: 数据不一致")
            for key, details in result['details'].items():
                if not details.get('data_match', False):
                    print(f"   - {key}: shape={details['original_shape']} vs {details['minimal_shape']}")
    else:
        print(f"⏭️  {output_filename}: 文件不存在，跳过验证")

if all(r['all_match'] for r in verification_results):
    print("\n✅ 所有抽样文件验证通过！数据提取正确。")
else:
    print("\n⚠️ 部分文件验证失败，请检查详细信息。")

## 7. 完成总结

In [ ]:
print("\n" + "="*60)
print("🎉 数据精简任务完成！")
print("="*60)
print(f"\n📂 精简数据位置: {OUTPUT_3D_MINIMAL_DIR}")
print(f"\n💾 下一步:")
print(f"   1. 在训练代码中使用精简数据")
print(f"   2. 只需加载3个key: data, region_mask, region_labels")
print(f"   3. 原始完整数据保持不变，可随时回退")
print("\n" + "="*60)